# SA-ZD-NIDS Project Walkthrough

This notebook provides a clear, end-to-end implementation guide for the **Self-Adaptive Zero-Day Aware Network Intrusion Detection System (SA-ZD-NIDS)** project.

What this notebook covers:
- project setup and environment checks
- loading and analyzing experiment outputs
- visualizing key performance metrics
- running a mini implementation pipeline on synthetic data
- how to run full training and experiment suites

## 1. Setup and Imports

This section configures paths and imports all required libraries.  
The code is intentionally explicit and commented for clarity.

In [ ]:
# Core Python utilities
from pathlib import Path
import sys
import json

# Data and plotting stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve project root from the current notebook execution directory
ROOT = Path.cwd().resolve()

# Add local source package path so "sa_zd_nids" imports work in Jupyter
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"Project root: {ROOT}")
print(f"Source path added: {SRC.exists()}")

Project root: C:\\Users\\Mehul\\Documents\\kiro\\network intrusion
Source path added: True


## 2. Hardware / Runtime Check

This confirms whether CUDA is available for GPU-enabled training.

In [ ]:
import platform
import torch

# Print runtime and hardware information in a compact, reproducible format
print("python", sys.version.split()[0])
print("platform", platform.platform())
print("cuda_available", torch.cuda.is_available())
print("cuda_device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

python 3.10.6
platform Windows-10-10.0.26200-SP0
cuda_available True
cuda_device NVIDIA GeForce RTX 4050 Laptop GPU


## 3. Load Existing Experiment Summary

This section reads the completed experiment suite output and builds a clean metrics table for analysis.

In [ ]:
# Path to the experiment summary generated by run_experiments.py
summary_path = ROOT / "evaluation" / "experiments" / "paper_run_20260401" / "summary.json"

if not summary_path.exists():
    raise FileNotFoundError(f"Summary file not found: {summary_path}")

# Load JSON summary and flatten the key metrics into a dataframe
summary = json.loads(summary_path.read_text(encoding="utf-8"))
rows = []
for method_name, payload in summary["experiments"].items():
    m = payload["metrics"]
    rows.append(
        {
            "method": method_name,
            "accuracy": m["accuracy"],
            "f1_macro": m["f1_macro"],
            "zero_day_detection_rate": m["zero_day_detection_rate"],
            "benign_fpr": m["zero_day_false_positive_rate_benign"],
            "latency_ms": m["avg_latency_ms_per_sample"],
            "cpu_percent": m["avg_cpu_percent"],
            "memory_mb": m["avg_memory_mb"],
            "num_drifts": m["num_drifts"],
        }
    )

metrics_df = pd.DataFrame(rows).sort_values("method").reset_index(drop=True)
print(metrics_df.to_string(index=False))

         method  accuracy  f1_macro  zero_day_detection_rate  benign_fpr  latency_ms  cpu_percent   memory_mb  num_drifts
     static_xgb  0.967489  0.194894                 0.000000    0.000000    0.006649    99.881745  637.970893           0
      static_ae  0.801757  0.155875                 0.667605    0.045575    0.004370    13.777631 1147.788151           0
hybrid_adaptive  0.967652  0.182953                 0.580066    0.000013    0.011008    99.843062 1108.793750           0


In [ ]:
# Visual comparison of key detection metrics across baselines
plot_df = metrics_df.set_index("method")[["accuracy", "f1_macro", "zero_day_detection_rate"]]

ax = plot_df.plot(kind="bar", figsize=(10, 5), rot=15)
ax.set_title("SA-ZD-NIDS: Core Detection Metrics by Method")
ax.set_ylabel("Score")
ax.set_xlabel("Method")
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## 4. Mini End-to-End Implementation (Synthetic Data)

This is a small, fully reproducible implementation path that mirrors the project flow:
1. create synthetic labeled traffic
2. preprocess + split chronologically
3. train classifier + autoencoder
4. run streaming engine
5. inspect resulting metrics

Use this for fast debugging before launching full dataset experiments.

In [ ]:
from sa_zd_nids.data.preprocessing import DataPreprocessor
from sa_zd_nids.models.classifier import KnownAttackClassifier
from sa_zd_nids.models.autoencoder import ZeroDayAutoencoder
from sa_zd_nids.streaming.engine import StreamingEngine

# 1) Generate synthetic network-flow style data
rng = np.random.default_rng(42)
n = 800
X = rng.normal(size=(n, 12)).astype(float)
labels = np.array(["BENIGN"] * n, dtype=object)
labels[rng.choice(n, size=n // 8, replace=False)] = "ATTACK"
timestamps = np.arange(n)

synthetic_df = pd.DataFrame(X, columns=[f"f{i}" for i in range(X.shape[1])])
synthetic_df["timestamp"] = timestamps
synthetic_df["label"] = labels

# 2) Lightweight config for notebook-scale execution
mini_cfg = {
    "project": {"random_state": 42},
    "data": {
        "label_col": "label",
        "timestamp_col": "timestamp",
        "scale": "standard",
        "test_size": 0.2,
        "val_size": 0.1,
        "benign_label": "BENIGN",
    },
    "features": {"use_mutual_info": False, "impute_strategy": "median"},
    "classifier": {
        "model_type": "xgboost",
        "use_gpu": False,
        "confidence_threshold": 0.7,
        "xgboost": {
            "n_estimators": 40,
            "learning_rate": 0.1,
            "max_depth": 4,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "reg_lambda": 1.0,
        },
        "random_forest": {"n_estimators": 40, "max_depth": 8},
    },
    "anomaly": {
        "use_gpu": False,
        "hidden_dims": [32, 16, 8],
        "epochs": 2,
        "batch_size": 64,
        "learning_rate": 0.001,
        "threshold_percentile": 95,
    },
    "drift": {
        "adwin_delta": 0.01,
        "window_size": 150,
        "min_adapt_samples": 80,
        "confidence_low_threshold": 0.55,
        "accuracy_weight": 0.6,
        "confidence_weight": 0.3,
        "feature_shift_weight": 0.1,
        "past_replay_fraction": 0.3,
        "adapt_validation_fraction": 0.2,
        "adapt_accept_tolerance": 0.0,
    },
    "streaming": {"batch_size": 100, "max_batches": None, "adaptation_history_keep": 2},
    "logging": {
        "output_csv": "logs/notebook_stream_predictions.csv",
        "events_jsonl": "logs/notebook_events.jsonl",
        "metrics_json": "logs/notebook_metrics.json",
    },
}

# 3) Fit preprocessing and train both model branches
pre = DataPreprocessor(mini_cfg)
prepared = pre.fit_transform(synthetic_df)

clf = KnownAttackClassifier(mini_cfg)
clf_art = clf.fit(prepared.X_train, prepared.y_train, prepared.X_val, prepared.y_val)

ae = ZeroDayAutoencoder(mini_cfg)
train_normal = prepared.X_train[prepared.y_train == "BENIGN"]
val_normal = prepared.X_val[prepared.y_val == "BENIGN"]
ae.fit(train_normal, val_normal)

# 4) Run streaming inference on the chronological test split
engine = StreamingEngine(mini_cfg, clf, ae)
artifacts = engine.run(
    X=prepared.X_test,
    y=prepared.y_test,
    timestamps=prepared.timestamps_test,
    benign_label="BENIGN",
    known_labels=clf_art.labels,
)

# 5) Display a compact result summary
mini_metrics = {
    "accuracy": artifacts.metrics.get("accuracy"),
    "f1_macro": artifacts.metrics.get("f1_macro"),
    "zero_day_detection_rate": artifacts.metrics.get("zero_day_detection_rate"),
    "zero_day_false_positive_rate_benign": artifacts.metrics.get("zero_day_false_positive_rate_benign"),
    "avg_latency_ms_per_sample": artifacts.metrics.get("avg_latency_ms_per_sample"),
}
pd.Series(mini_metrics, name="mini_pipeline_metrics")

## 5. Inspect Prediction Samples from Real Run

This cell reads the first few rows from your real hybrid predictions file.

In [ ]:
pred_path = ROOT / "evaluation" / "experiments" / "paper_run_20260401" / "hybrid_adaptive" / "predictions.csv"
pred_head = pd.read_csv(pred_path, nrows=5)
print(pred_head.to_string(index=False))

                                                      timestamp  true_label  prediction  confidence  reconstruction_error  drift_flag  adapted  cpu_percent   memory_mb  latency_ms_per_sample
timestamp    936232\ntimestamp    936232\nName: 0, dtype: int64 SSH-Patator SSH-Patator    0.999748                   NaN       False    False         55.3 1137.777344               0.026053
timestamp    936232\ntimestamp    936232\nName: 1, dtype: int64    DoS Hulk    DoS Hulk    0.999901                   NaN       False    False         55.3 1137.777344               0.026053
timestamp    936233\ntimestamp    936233\nName: 2, dtype: int64      BENIGN      BENIGN    0.998132                   NaN       False    False         55.3 1137.777344               0.026053
timestamp    936233\ntimestamp    936233\nName: 3, dtype: int64    DoS Hulk    DoS Hulk    0.999901                   NaN       False    False         55.3 1137.777344               0.026053
timestamp    936234\ntimestamp    936234\nNam

## 6. Full Project Commands

Use these commands for full-scale runs outside the notebook:

```bash
python train.py --config config.yaml
python stream.py --config config.yaml
python run_experiments.py --config config_fast.yaml --outdir evaluation/experiments/paper_run_20260401
```

If you want, this notebook can also be extended to automatically call those commands using `subprocess`.